# Group G

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
# This might be needed in order to get plots to display in the exported HTML for submission.
# In any case, please double check that plots display properly before you submit.
import plotly.io as pio
pio.renderers.default='notebook'
df = pd.read_csv("WeatherEvents_Jan2016-Dec2022.csv")

## Picco Massimo Storico di Precipitazioni per Ogni Aeroporto (2016-2022)

Il grafico mostra il picco massimo di precipitazioni (in pollici) registrato storicamente da ciascuna stazione aeroportuale presente nel dataset. La visualizzazione evidenzia inoltre la distribuzione omogenea dei punti di rilevamento sull'intero territorio degli Stati Uniti.

In [ ]:
df_ordinato = df.sort_values(by='Precipitation(in)', ascending=False)
df_aeroporti = df_ordinato.drop_duplicates(subset=['LocationLat', 'LocationLng'])

fig = px.scatter_geo(df_aeroporti, 
                     lat="LocationLat", 
                     lon="LocationLng", 
                     scope='usa',
                     hover_name="City",
                    color_continuous_scale="Turbo",
                    title="Picco Massimo Storico di Precipitazioni per Ogni Aeroporto (2016-2022)")
fig.update_layout(hovermode=False)
fig.show()

## Distribuzione delle Precipitazioni Giornaliere in Texas - Uragano Harvey (Agosto 2017)

Il grafico mostra la distribuzione delle precipitazioni (in millimetri) registrate in Texas nel mese di agosto 2017. L'elevata concentrazione di valori estremi (outliers) evidenzia chiaramente l'impatto anomalo causato dal passaggio dell'Uragano Harvey sullo Stato.

In [ ]:
#Analisi Uragano Harvey
df['StartTime(UTC)'] = pd.to_datetime(df['StartTime(UTC)'])

df_texas = df[df['State'] == 'TX'].copy()

df_texas['Year'] = df_texas['StartTime(UTC)'].dt.year
df_texas['Month'] = df_texas['StartTime(UTC)'].dt.month
df_texas['Day'] = df_texas['StartTime(UTC)'].dt.day

df_texas = df_texas[(df_texas['Year']==2017) & (df_texas['Month'] == 8)]

df_prec = df_texas[df_texas['Precipitation(in)'] > 0]

df_prec['Precipitation(mm)'] = df_prec['Precipitation(in)'] * 25.4;

fig = px.box(df_prec,
            x='Day',
            y='Precipitation(mm)',
            title="Distribuzione delle Precipitazioni Giornaliere in Texas - Uragano Harvey (Agosto 2017)",
            labels={'Day': 'Giorno', 'Precipitation(in)': 'Precipitazioni (pollici)'},
            )
fig.update_layout(xaxis=dict(tickmode='linear', dtick=1))

fig.add_vline(
    x=27,                           # Posizione sull'asse X (27 agosto) (picco massimo)
    line_width=2,                   
    line_dash="dash",               
    line_color="red",               
    annotation_text="Picco Harvey (27 Ago)", 
    annotation_position="top left", 
    annotation_font_color="red",
    opacity=0.5
)


fig.show()


## Picco Massimo di Precipitazioni per Stato negli USA (Agosto 2017)

Il grafico mostra il picco massimo di precipitazioni per Stato nell'agosto 2017. I dati evidenziano chiaramente l'impatto eccezionale dell'Uragano Harvey sul Texas rispetto al resto del Paese.

In [ ]:
df['StartTime(UTC)'] = pd.to_datetime(df['StartTime(UTC)'])
df_agosto_2017 = df[(df['StartTime(UTC)'].dt.year == 2017) & 
                    (df['StartTime(UTC)'].dt.month == 8) & 
                    (df['Precipitation(in)'] > 0)].copy()

df_agosto_2017['Precipitation(mm)'] = df_agosto_2017['Precipitation(in)'] * 25.4

#Anomalia trovata nello stato di NY 1500mm (Impossibile) e 520mm (Impossibile), ho controllato nessun evento rilevante quel giorno

#Abbassa la soglia di pulizia per escludere l'ulteriore anomalia di 520 mm
df_agosto_2017 = df_agosto_2017[df_agosto_2017['Precipitation(mm)'] <= 450]

precipitazioni_stato = df_agosto_2017.groupby('State')['Precipitation(mm)'].max().reset_index()

fig = px.choropleth(
    precipitazioni_stato,
    locations='State',                 
    locationmode='USA-states',         
    color='Precipitation(mm)',         
    color_continuous_scale='Reds',
    range_color=[0, 430],             #Evita anomalie
    scope='usa',                       
    title="Picco Massimo di Precipitazioni per Stato negli USA (Agosto 2017)",
    labels={'Precipitation(mm)': 'Precipitazione Massima (mm)'}
)

fig.show()

## Spostamento e Intensità dell'Uragano Harvey (Agosto 2017)

Il grafico animato mappa lo spostamento e l'intensità delle precipitazioni nell'agosto 2017. La visualizzazione permette di seguire chiaramente l'andamento dell'Uragano Harvey lungo il suo percorso, evidenziando come il picco massimo del fenomeno si registri esattamente il 27 agosto.

In [ ]:
df['StartTime(UTC)'] = pd.to_datetime(df['StartTime(UTC)'])

df_agosto_2017 = df[(df['StartTime(UTC)'].dt.year == 2017) & (df['StartTime(UTC)'].dt.month == 8)]
df_agosto_2017 = df_agosto_2017[df_agosto_2017['Severity'] == 'Heavy' ]

df_agosto_2017['Day'] = df_agosto_2017['StartTime(UTC)'].dt.day

#Pulizia del dataset
df_agosto_2017['Precipitation(mm)'] = df_agosto_2017['Precipitation(in)'] * 25.4
df_agosto_2017 = df_agosto_2017[(df_agosto_2017['Precipitation(mm)'] >= 50) & (df_agosto_2017['Precipitation(mm)'] < 500) & (df_agosto_2017['Type'] == 'Rain')]
df_agosto_2017 = df_agosto_2017[df_agosto_2017['State'] != 'NY']

df_agosto_2017 = df_agosto_2017.sort_values(by='Day')


fig = px.scatter_geo(
    df_agosto_2017,
    lat="LocationLat",
    lon="LocationLng",
    color="Precipitation(mm)",        
    size="Precipitation(mm)",         
    hover_name="City",                
    animation_frame="Day",        
    scope='usa',                      
    color_continuous_scale="Reds",   
    range_color=[0, df_agosto_2017['Precipitation(mm)'].max()],
    title="Spostamento e Intensità dell'Uragano Harvey (Agosto 2017)",
    height=700,
    template="plotly_dark")


fig.show()

## Numero Totale di Eventi per Tipo (per Anno)

Il grafico animato mostra il numero totale di eventi meteorologici per tipologia, scorrendo anno per anno. L'utilizzo di una scala logaritmica sull'asse orizzontale permette di confrontare chiaramente fenomeni molto rari (come la grandine) con eventi estremamente frequenti (come pioggia e nebbia), garantendo la visibilità e la proporzione di tutti i dati.

Il grafico mostra che il numero di eventi meteorologici non cambia molto da un anno all'altro: 6 anni sono infatti troppo pochi per notare gli effetti del cambiamento climatico. La classifica rimane identica, nonostante qualche piccola differenza nel numero di registrazioni fra i vari anni. Nebbia e pioggia sono costantemente gli eventi più comuni ogni anno, mentre grandine e tempeste restano sempre in fondo alla lista.

I grafici precedenti mostrano l'andamento generale, ma non permettono di dare una risposta chiara a una domanda precisa: piove o nevica più di una volta nello stesso posto?

In [ ]:
df['StartTime(UTC)'] = pd.to_datetime(df['StartTime(UTC)'])
df['Year'] = df['StartTime(UTC)'].dt.year

df_filtered = df[df['Type'] != 'Precipitation']

df_counts = df_filtered.groupby(['Year', 'Type']).size().reset_index(name='Count')
df_counts = df_counts.sort_values(by=['Year', 'Count'])

max_count = df_counts['Count'].max()

fig = px.bar(
    df_counts,
    x='Count',
    y='Type',
    animation_frame='Year',       
    orientation='h',              
    title="Numero Totale di Eventi per Tipo (per Anno)",
    labels={'Count': 'Numero Totale (Scala Log)', 'Type': 'Tipo di Evento'},
    log_x=True,                   # Abilita l'asse X logaritmico
    range_x=[1, max_count * 1.5],
    color='Type',                 
)

fig.update_yaxes(categoryorder='total ascending')
fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 1200 
fig.update_layout(legend_traceorder="reversed")

fig.show()